# DWH_FactorLake_BatchIngest_V0_1

One-click batch raw factor-lake ingest notebook for AkShare factor/data-source candidates derived from `Factor_test.ipynb` reference API set.


In [ ]:
# Colab-friendly setup
import os
import json
import uuid
import traceback
import inspect
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

try:
    import akshare as ak
except Exception as e:
    raise RuntimeError("akshare is required. Please run: !pip -q install akshare pyarrow") from e


In [ ]:
# Canonical roots
RAW_ROOT = Path('/content/drive/MyDrive/a_share_quant_cache/raw/factor_lake/akshare/v1')
CATALOG_ROOT = Path('/content/drive/MyDrive/a_share_quant_cache/catalog/factor_lake/akshare/v1')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/a_share_quant_cache/artifacts/factor_lake_ingest')

for p in [RAW_ROOT, CATALOG_ROOT, ARTIFACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)


In [ ]:
# API reference set extracted from Factor_test.ipynb candidate/probe lists (phase-18A migration contract)
FACTOR_TEST_API_REFERENCE = sorted(set(['stock_zh_a_hist', 'stock_individual_info_em', 'index_zh_a_hist', 'stock_zh_index_hist_csindex', 'stock_zh_index_daily', 'stock_zh_index_daily_em', 'index_hist_cni', 'index_detail_hist_cni', 'stock_financial_analysis_indicator', 'stock_yjyg_em', 'stock_yysj_em', 'stock_zh_a_disclosure_relation_cninfo', 'stock_margin_sse', 'stock_margin_detail_sse', 'stock_margin_szse', 'stock_margin_detail_szse', 'stock_margin_underlying_info_szse', 'macro_china_market_margin_sz', 'stock_industry_category_cninfo', 'stock_industry_change_cninfo', 'stock_industry_clf_hist_sw', 'sw_index_first_info', 'sw_index_second_info', 'sw_index_third_info', 'sw_index_third_cons', 'index_component_sw', 'index_hist_sw', 'index_realtime_sw', 'stock_board_industry_name_ths', 'stock_board_industry_cons_ths', 'stock_board_industry_index_ths', 'stock_board_industry_info_ths', 'stock_board_industry_summary_ths', 'stock_board_concept_name_ths', 'stock_board_concept_cons_ths', 'stock_board_concept_index_ths', 'stock_board_concept_info_ths', 'stock_board_concept_summary_ths', 'stock_board_industry_name_em', 'stock_board_industry_cons_em', 'stock_board_industry_hist_em', 'stock_board_concept_name_em', 'stock_board_concept_cons_em', 'stock_board_concept_hist_em', 'index_stock_cons_csindex', 'index_stock_cons_weight_csindex', 'stock_zh_a_gdhs', 'stock_zh_a_gdhs_detail_em', 'stock_gdfx_free_holding_analyse_em', 'stock_gdfx_holding_analyse_em', 'stock_gpzy_pledge_ratio_em', 'stock_gpzy_pledge_ratio_detail_em', 'stock_gpzy_industry_data_em', 'stock_gpzy_profile_em', 'stock_fhps_em', 'stock_history_dividend', 'stock_history_dividend_detail', 'stock_restricted_release_queue_em', 'stock_restricted_release_summary_em', 'stock_restricted_release_detail_em', 'stock_dzjy_sctj', 'stock_dzjy_mrmx', 'stock_dzjy_mrtj', 'stock_dzjy_hyyybtj', 'stock_lhb_detail_em', 'stock_lhb_stock_statistic_em', 'stock_lhb_jgmmtj_em', 'stock_lhb_hyyyb_em', 'stock_lhb_yybph_em', 'stock_jgdy_tj_em', 'stock_jgdy_detail_em']))

FAMILY_MAP = {
    'market_price': {'stock_zh_a_hist', 'stock_individual_info_em'},
    'index_market': {'index_zh_a_hist','stock_zh_index_hist_csindex','stock_zh_index_daily','stock_zh_index_daily_em','index_hist_cni','index_detail_hist_cni','index_component_sw','index_hist_sw','index_realtime_sw','index_stock_cons_csindex','index_stock_cons_weight_csindex'},
    'financial_fundamental': {'stock_financial_analysis_indicator','stock_yjyg_em','stock_yysj_em'},
    'disclosure_ir': {'stock_zh_a_disclosure_relation_cninfo','stock_jgdy_tj_em','stock_jgdy_detail_em'},
    'margin_leverage': {'stock_margin_sse','stock_margin_detail_sse','stock_margin_szse','stock_margin_detail_szse','stock_margin_underlying_info_szse','macro_china_market_margin_sz'},
    'industry_concept': {'stock_industry_category_cninfo','stock_industry_change_cninfo','stock_industry_clf_hist_sw','sw_index_first_info','sw_index_second_info','sw_index_third_info','sw_index_third_cons','stock_board_industry_name_ths','stock_board_industry_cons_ths','stock_board_industry_index_ths','stock_board_industry_info_ths','stock_board_industry_summary_ths','stock_board_concept_name_ths','stock_board_concept_cons_ths','stock_board_concept_index_ths','stock_board_concept_info_ths','stock_board_concept_summary_ths','stock_board_industry_name_em','stock_board_industry_cons_em','stock_board_industry_hist_em','stock_board_concept_name_em','stock_board_concept_cons_em','stock_board_concept_hist_em'},
    'event_ownership': {'stock_zh_a_gdhs','stock_zh_a_gdhs_detail_em','stock_gdfx_free_holding_analyse_em','stock_gdfx_holding_analyse_em','stock_gpzy_pledge_ratio_em','stock_gpzy_pledge_ratio_detail_em','stock_gpzy_industry_data_em','stock_gpzy_profile_em','stock_fhps_em','stock_history_dividend','stock_history_dividend_detail','stock_restricted_release_queue_em','stock_restricted_release_summary_em','stock_restricted_release_detail_em','stock_dzjy_sctj','stock_dzjy_mrmx','stock_dzjy_mrtj','stock_dzjy_hyyybtj','stock_lhb_detail_em','stock_lhb_stock_statistic_em','stock_lhb_jgmmtj_em','stock_lhb_hyyyb_em','stock_lhb_yybph_em'},
}

def family_of(api_name: str) -> str:
    for fam, apis in FAMILY_MAP.items():
        if api_name in apis:
            return fam
    return 'disclosure_ir'

def defaults_for(api_name: str):
    # Generic lightweight default args; signature filtering will remove unsupported kwargs.
    return {
        'symbol': '000001', 'stock': '000001', 'code': '000001', 'index_code': '000300',
        'start_date': '20200101', 'end_date': '20201231', 'date': '20200101',
        'indicator': '按报告期', 'period': 'daily', 'market': '全部', 'adjust': '',
        'board': '行业', 'sw_level': '1',
    }

FACTOR_SOURCE_REGISTRY = []
for api in FACTOR_TEST_API_REFERENCE:
    FACTOR_SOURCE_REGISTRY.append({
        'api_name': api,
        'source_family': family_of(api),
        'source_type': 'akshare_raw',
        'case_id': 'default',
        'kwargs': defaults_for(api),
        'expected_frequency': 'unknown',
        'date_col_hint': '日期',
        'symbol_col_hint': '代码',
        'pit_relevance': 'unknown',
        'research_value': 'unknown',
        'enabled': True,
        'notes': 'Migrated from Factor_test reference API set',
    })


In [ ]:
def filter_kwargs(func, kwargs):
    sig = inspect.signature(func)
    allowed = set(sig.parameters.keys())
    return {k: v for k, v in kwargs.items() if k in allowed}

def write_frame(df: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = out_dir / 'data.parquet'
    csv_path = out_dir / 'data.csv'
    try:
        df.to_parquet(parquet_path, index=False)
        return str(parquet_path), 'parquet'
    except Exception:
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        return str(csv_path), 'csv_fallback'

def run_case(entry):
    api_name = entry['api_name']
    now = datetime.now(timezone.utc).isoformat()
    base = {k: entry.get(k) for k in entry if k != 'kwargs'}
    base.update({'attempted_at_utc': now, 'status': None, 'rows': None, 'error': None, 'output_path': None, 'output_format': None})

    if not entry.get('enabled', True):
        base['status'] = 'skipped'
        return base

    func = getattr(ak, api_name, None)
    if func is None:
        base['status'] = 'missing'
        base['error'] = 'function_not_found_in_akshare'
        return base

    kwargs = filter_kwargs(func, entry.get('kwargs', {}))
    try:
        res = func(**kwargs)
        if isinstance(res, pd.DataFrame):
            if res.empty:
                base['status'] = 'empty'
                base['rows'] = 0
                return base
            out_dir = RAW_ROOT / f"family={entry['source_family']}" / f"api={api_name}" / f"case={entry['case_id']}"
            out_path, fmt = write_frame(res, out_dir)
            meta = dict(base)
            meta.update({'status': 'success', 'rows': int(len(res)), 'output_path': out_path, 'output_format': fmt, 'used_kwargs': kwargs})
            with open(out_dir / 'metadata.json', 'w', encoding='utf-8') as f:
                json.dump(meta, f, ensure_ascii=False, indent=2)
            base.update({'status': 'success', 'rows': int(len(res)), 'output_path': out_path, 'output_format': fmt})
            return base
        base['status'] = 'failed'
        base['error'] = f'non_dataframe_return:{type(res).__name__}'
        return base
    except Exception as e:
        base['status'] = 'failed'
        base['error'] = f"{type(e).__name__}: {e}"
        return base


In [ ]:
# One-click batch execution
RUN_ALL = True

if RUN_ALL:
    registry_df = pd.DataFrame(FACTOR_SOURCE_REGISTRY)
    inventory_rows = [run_case(entry) for entry in FACTOR_SOURCE_REGISTRY]
    inv_df = pd.DataFrame(inventory_rows)

    ref_set = set(FACTOR_TEST_API_REFERENCE)
    reg_set = set(registry_df['api_name'].unique())
    unregistered = sorted(ref_set - reg_set)

    coverage = {
        'total_apis_found_from_factor_test_reference': len(ref_set),
        'total_apis_registered': len(reg_set),
        'total_cases_generated': int(len(registry_df)),
        'total_cases_attempted': int((inv_df['status'] != 'skipped').sum()),
        'success_count': int((inv_df['status'] == 'success').sum()),
        'failed_count': int((inv_df['status'] == 'failed').sum()),
        'empty_count': int((inv_df['status'] == 'empty').sum()),
        'missing_function_count': int((inv_df['status'] == 'missing').sum()),
        'skipped_count': int((inv_df['status'] == 'skipped').sum()),
        'unregistered_api_names': '|'.join(unregistered),
        'registered_coverage_ratio': 0 if len(ref_set) == 0 else len(reg_set & ref_set) / len(ref_set),
    }

    registry_df.to_csv(CATALOG_ROOT / 'factor_source_registry.csv', index=False, encoding='utf-8-sig')
    inv_df.to_csv(CATALOG_ROOT / 'factor_ingest_inventory.csv', index=False, encoding='utf-8-sig')
    inv_df[inv_df['status'] == 'failed'].to_csv(CATALOG_ROOT / 'factor_ingest_failed.csv', index=False, encoding='utf-8-sig')
    inv_df[inv_df['status'] == 'empty'].to_csv(CATALOG_ROOT / 'factor_ingest_empty.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame([coverage]).to_csv(CATALOG_ROOT / 'factor_ingest_coverage_report.csv', index=False, encoding='utf-8-sig')

    manifest = {
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'raw_root': str(RAW_ROOT),
        'catalog_root': str(CATALOG_ROOT),
        'total_cases': int(len(registry_df)),
        'coverage': coverage,
    }
    with open(ARTIFACT_ROOT / 'factor_lake_ingest_manifest.json', 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print('Batch ingest finished')
    print(pd.DataFrame([coverage]).T)
